# Notebook 6: Classification Models — Predicting Rate Direction
## Federal Reserve Interest Rate Prediction

**Target Classes:** Increase / Decrease / No_Change

**Models:** Logistic Regression, Naive Bayes, SVM, Decision Tree, Random Forest, Gradient Boosting, XGBoost

**Evaluation Metrics:**
- Accuracy, Precision, Recall, F1-Score (weighted)
- Confusion Matrix
- ROC-AUC (one-vs-rest)
- Learning Curves


In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

plt.style.use('seaborn-v0_8-whitegrid')
PALETTE = ['#2196F3','#F44336','#4CAF50','#FF9800','#9C27B0',
           '#00BCD4','#E91E63','#795548','#607D8B','#FF5722']
sns.set_palette(PALETTE)

DATA_PATH = r"d:/Projects/ML website/ML-Project/App/Tabs/Datasets/finaldataset.csv"
OUT_PATH  = r"d:/Projects/ML website/ML-Project/ml_analysis/outputs"

from sklearn.model_selection import (train_test_split, cross_val_score,
                                     learning_curve, TimeSeriesSplit)
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                              roc_auc_score, roc_curve, f1_score, precision_score, recall_score)
from sklearn.preprocessing import label_binarize, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
import xgboost as xgb


In [ ]:
import pickle
with open(f"{OUT_PATH}/results/preprocessed_data.pkl", "rb") as f:
    data = pickle.load(f)

X_scaled    = data['X_scaled']
y_cls       = data['y_cls']
label_names = data['label_names']
feat_cols   = data['feature_cols']

X_tr, X_te, y_tr, y_te = train_test_split(X_scaled, y_cls, test_size=0.2,
                                            random_state=42, stratify=y_cls)
tscv = TimeSeriesSplit(n_splits=5)

print(f"Classes: {label_names}")
print(f"Class distribution: {dict(zip(label_names, np.bincount(y_cls)))}")
print(f"Baseline (majority class): {max(np.bincount(y_cls))/len(y_cls):.3f}")
print(f"Baseline (random 3-class): {1/3:.3f}")


In [ ]:
def eval_cls(model, name):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te) if hasattr(model,'predict_proba') else None
    acc  = accuracy_score(y_te, y_pred)
    f1   = f1_score(y_te, y_pred, average='weighted')
    cv_s = cross_val_score(model, X_scaled, y_cls, cv=tscv, scoring='accuracy')
    print(f"\n{'='*60}")
    print(f"Model: {name}")
    print(f"Accuracy: {acc:.4f} | F1: {f1:.4f} | CV: {cv_s.mean():.4f}±{cv_s.std():.4f}")
    print(classification_report(y_te, y_pred, target_names=label_names))
    return y_pred, y_prob, {'Accuracy':acc,'F1':f1,'CV':cv_s.mean()}

def plot_cm(y_true, y_pred, name):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=label_names, yticklabels=label_names)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.set_title(f'{name} — Confusion Matrix', fontweight='bold')
    plt.tight_layout(); plt.show()

def plot_roc(y_true, y_prob, name):
    y_bin = label_binarize(y_true, classes=[0,1,2])
    fig, ax = plt.subplots(figsize=(7, 6))
    for i, (cls, col) in enumerate(zip(label_names,['#2196F3','#F44336','#4CAF50'])):
        fpr, tpr, _ = roc_curve(y_bin[:,i], y_prob[:,i])
        auc = roc_auc_score(y_bin[:,i], y_prob[:,i])
        ax.plot(fpr, tpr, color=col, linewidth=2, label=f'{cls} (AUC={auc:.3f})')
    ax.plot([0,1],[0,1],'k--'); ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title(f'ROC Curves — {name}', fontweight='bold'); ax.legend()
    plt.tight_layout(); plt.show()

cls_results = {}


## 1. Logistic Regression

In [ ]:
log_reg = LogisticRegression(max_iter=2000, C=1.0, solver='lbfgs', random_state=42)
y_pred_lr, y_prob_lr, cls_results['Logistic Regression'] = eval_cls(log_reg, 'Logistic Regression')
plot_cm(y_te, y_pred_lr, 'Logistic Regression')
plot_roc(y_te, y_prob_lr, 'Logistic Regression')


## 2. Naive Bayes

In [ ]:
gnb = GaussianNB()
y_pred_nb, y_prob_nb, cls_results['Naive Bayes'] = eval_cls(gnb, 'Naive Bayes')
plot_cm(y_te, y_pred_nb, 'Naive Bayes')
plot_roc(y_te, y_prob_nb, 'Naive Bayes')


## 3. Support Vector Machine

In [ ]:
svc = SVC(kernel='rbf', C=10, gamma='scale', probability=True, random_state=42)
y_pred_svm, y_prob_svm, cls_results['SVM'] = eval_cls(svc, 'SVM (RBF)')
plot_cm(y_te, y_pred_svm, 'SVM')
plot_roc(y_te, y_prob_svm, 'SVM')


## 4. Decision Tree (with Depth Tuning)

In [ ]:
depths = range(1, 20)
dt_cv_accs = [cross_val_score(DecisionTreeClassifier(max_depth=d, random_state=42),
                               X_scaled, y_cls, cv=tscv, scoring='accuracy').mean()
              for d in depths]
best_d = depths[np.argmax(dt_cv_accs)]
print(f"Best depth: {best_d}")

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(depths, dt_cv_accs, 'r-o')
ax.axvline(best_d, color='green', linestyle='--', label=f'Best depth={best_d}')
ax.set_xlabel('Depth'); ax.set_ylabel('CV Accuracy')
ax.set_title('Decision Tree: Depth Tuning', fontweight='bold'); ax.legend()
plt.tight_layout(); plt.show()

dtc = DecisionTreeClassifier(max_depth=best_d, random_state=42)
y_pred_dt, y_prob_dt, cls_results['Decision Tree'] = eval_cls(dtc, f'Decision Tree (d={best_d})')
plot_cm(y_te, y_pred_dt, 'Decision Tree')


## 5. Random Forest

In [ ]:
rfc = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
y_pred_rf, y_prob_rf, cls_results['Random Forest'] = eval_cls(rfc, 'Random Forest')
rfc.fit(X_tr, y_tr)
plot_cm(y_te, y_pred_rf, 'Random Forest')
plot_roc(y_te, y_prob_rf, 'Random Forest')

# Feature importance
fi = pd.Series(rfc.feature_importances_, index=feat_cols).sort_values(ascending=False).head(20)
fig, ax = plt.subplots(figsize=(10, 6))
fi.plot.bar(ax=ax, color=PALETTE[:20])
ax.set_title('Random Forest Feature Importance (Classification)', fontweight='bold')
ax.tick_params(axis='x', rotation=45); plt.tight_layout(); plt.show()


## 6. Gradient Boosting & XGBoost

In [ ]:
gbc = GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42)
y_pred_gb, y_prob_gb, cls_results['Gradient Boosting'] = eval_cls(gbc, 'Gradient Boosting')
plot_cm(y_te, y_pred_gb, 'Gradient Boosting')
plot_roc(y_te, y_prob_gb, 'Gradient Boosting')


In [ ]:
xgb_cls = xgb.XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=4,
                              subsample=0.8, colsample_bytree=0.8, random_state=42,
                              eval_metric='mlogloss', verbosity=0)
y_pred_xgb, y_prob_xgb, cls_results['XGBoost'] = eval_cls(xgb_cls, 'XGBoost')
xgb_cls.fit(X_tr, y_tr)
plot_cm(y_te, y_pred_xgb, 'XGBoost')
plot_roc(y_te, y_prob_xgb, 'XGBoost')


## 7. Final Comparison

In [ ]:
cls_df = pd.DataFrame(cls_results).T.sort_values('Accuracy', ascending=False)
display(cls_df.round(4).style.highlight_max(color='lightgreen'))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
models = cls_df.index
axes[0].bar(models, cls_df['Accuracy'], color=PALETTE[:len(models)])
axes[0].axhline(1/3, color='red', linestyle='--', label='Random baseline (33%)')
axes[0].set_xticklabels(models, rotation=35, ha='right')
axes[0].set_title('Accuracy Comparison', fontweight='bold'); axes[0].legend()

axes[1].bar(models, cls_df['F1'], color=PALETTE[:len(models)])
axes[1].set_xticklabels(models, rotation=35, ha='right')
axes[1].set_title('Weighted F1 Score', fontweight='bold')
plt.suptitle('Classification Models Performance', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()
